# Guardrails AI — Hands-On Notebook

**Guardrails AI** lets you put *validators* around a language model's input and output.
You compose validators into a **Guard** that sits in your app's critical path: text flows
in, the validators run, and — depending on your policy — the Guard passes it, fixes it,
blocks it, or asks the model to try again.

```
user ─▶ [ Guard: validator₁ · validator₂ · … ] ─▶ LLM ─▶ [ same Guard on output ] ─▶ user
```

### The four concepts

| Term | What it is |
|------|-----------|
| **Validator** | A single check on text — regex, length, PII, toxicity, … |
| **Guard** | One or more validators wrapped together; you run text (or a whole LLM call) through it |
| **Hub** | The registry where validators are distributed: `guardrails hub install hub://guardrails/<name>` |
| **OnFailAction** | What a validator does when it fails: `noop` / `exception` / `fix` / `filter` / `reask` |

This notebook walks through 6 steps, from a trivial regex Guard to production validators
(PII, toxicity, competitor mentions) wrapped around a real Groq LLM call.

## Setup (one-time)

1. **Hub token** (free): create one at <https://hub.guardrailsai.com/tokens>, then run
   `guardrails configure` and paste it. This writes `~/.guardrailsrc`.
2. **Install the validators** used below. Each becomes importable from `guardrails.hub`
   only *after* installation:
   ```bash
   guardrails hub install hub://guardrails/regex_match
   guardrails hub install hub://guardrails/valid_length
   guardrails hub install hub://guardrails/toxic_language
   guardrails hub install hub://guardrails/detect_pii
   guardrails hub install hub://guardrails/competitor_check
   ```
   > **Windows note:** the installer prints a ✅ emoji that the default console encoding
   > can't render, crashing with `'charmap' codec can't encode character`. Prefix the command
   > with `PYTHONUTF8=1` (bash) or `$env:PYTHONUTF8=1;` (PowerShell) to fix it.
   >
   > Installs create a `.guardrails/hub_registry.json` **in the current folder**, and imports
   > resolve against it — so run this notebook from the same folder you installed into.
3. **LLM key**: Steps 4–5 call Groq through LiteLLM. Put `GROQ_API_KEY` in a `.env` file.

## Step 1 — Your first Guard (no LLM)

We start with zero LLM involvement to isolate what a Guard *is*. The rule: text must be a
single capitalized word, expressed as the regex `^[A-Z][a-z]*$`.

- `Guard()` creates an empty Guard.
- `.use(validator)` attaches a validator and returns the Guard.
- `RegexMatch` passes only if the text fully matches the pattern.
- `on_fail=OnFailAction.NOOP` is important: the **default** `on_fail` for most validators is
  `exception`, which *raises* on failure. We use `NOOP` so we can read a `True`/`False`
  verdict instead of catching an exception. (Failure actions are the topic of Step 2.)

In [1]:
from guardrails import Guard, OnFailAction
from guardrails.hub import RegexMatch  # importable only AFTER `guardrails hub install`

import warnings
warnings.filterwarnings("ignore")  # silence noisy dependency warnings for a clean notebook

In [2]:
# Build a Guard whose single rule is "one capitalized word".
guard = Guard().use(
    RegexMatch(regex="^[A-Z][a-z]*$", on_fail=OnFailAction.NOOP)
)

In [3]:
# .parse(text) runs the validators against `text`. NO LLM call happens here —
# it treats `text` as if it were model output and validates it.
result = guard.parse("Hello")
result

ValidationOutcome[TypeVar](call_id='2157874750288', raw_llm_output='Hello', validation_summaries=[], validated_output='Hello', reask=None, validation_passed=True, error=None)

`.parse()` returns a **`ValidationOutcome`**. The fields you'll use most:

| Field | Meaning |
|-------|---------|
| `validation_passed` | `True` if every validator passed |
| `validated_output` | the text the Guard returns (after any `fix`) |
| `raw_llm_output` | the original text before validation |
| `validation_summaries` | one entry per failed validator (empty when all pass) |

In [4]:
result.validation_passed

True

In [5]:
def check(text: str) -> None:
    """Validate `text` and print a PASS/FAIL line."""
    result = guard.parse(text)
    status = "PASS ✅" if result.validation_passed else "FAIL ❌"
    print(f"{status}  {text!r}")

In [6]:
check("Caesar")        # single capitalized word -> PASS
check("Caesar Salad")  # contains a space          -> FAIL
check("caesar")        # lowercase first letter    -> FAIL

PASS ✅  'Caesar'
FAIL ❌  'Caesar Salad'
FAIL ❌  'caesar'


## Step 2 — `OnFailAction`: what happens when a validator fails

A failing validator isn't the end — **you decide the consequence** with `on_fail=`. This is
the most important knob in Guardrails.

| Action | Behavior | When to use |
|--------|----------|-------------|
| `NOOP` | record the failure, return text unchanged | logging / observability |
| `EXCEPTION` | raise an error, stop the pipeline | production fail-closed: bad output must be blocked |
| `FIX` | auto-correct to a passing value (validator-specific) | sanitize instead of reject |
| `FILTER` | drop the failing value | structured output where one bad field shouldn't kill the rest |
| `REASK` | ask the LLM to try again | only meaningful with a model in the loop (Step 4) |

We demonstrate the first three with `ValidLength`, which passes when `min ≤ len(text) ≤ max`.

In [7]:
from guardrails.hub import ValidLength  # passes if min <= len(text) <= max

TEXT = "this string is definitely far too long for the limit"  # 52 chars, limit is 10

In [8]:
# 1) NOOP — never raises. You inspect the verdict yourself.
noop_guard = Guard().use(ValidLength(min=1, max=10, on_fail=OnFailAction.NOOP))
res = noop_guard.parse(TEXT)
print("NOOP      -> validation_passed:", res.validation_passed)   # False

NOOP      -> validation_passed: False


In [9]:
# 2) EXCEPTION — fail-closed. The go-to for production safety checks.
strict_guard = Guard().use(ValidLength(min=1, max=10, on_fail=OnFailAction.EXCEPTION))
try:
    strict_guard.parse(TEXT)
except Exception as e:
    print("EXCEPTION -> raised:", type(e).__name__)               # ValidationError

EXCEPTION -> raised: ValidationError


In [10]:
# 3) FIX — auto-corrects. ValidLength's fix truncates the text to `max` characters.
fix_guard = Guard().use(ValidLength(min=1, max=10, on_fail=OnFailAction.FIX))
fixed = fix_guard.parse(TEXT)
print("FIX       -> validation_passed:", fixed.validation_passed)   # True (it was corrected)
print("FIX       -> validated_output :", repr(fixed.validated_output))

FIX       -> validation_passed: True
FIX       -> validated_output : 'this strin'


## Step 3 — Stacking multiple validators

A Guard can hold several validators; the text must pass **all** of them.

> ⚠️ **Gotcha (important):** pass every validator to a **single** `.use(...)` call.
> Calling `.use()` twice — `Guard().use(A).use(B)` — does **not** stack them; the second call
> **replaces** the first, so `A` is silently dropped. Always write `Guard().use(A, B)`.

Rule below: a single capitalized word (regex) **and** 1–12 characters (length).

In [11]:
# CORRECT: both validators in one .use(...) call so BOTH are active.
guard = Guard().use(
    RegexMatch(regex="^[A-Z][a-z]*$", on_fail=OnFailAction.NOOP),
    ValidLength(min=1, max=12, on_fail=OnFailAction.NOOP),
)

def check(text: str) -> None:
    res = guard.parse(text)
    print(f"{'PASS ✅' if res.validation_passed else 'FAIL ❌'}  {text!r}")

check("Caesar")               # word ✅ + length ✅      -> PASS
check("Supercalifragilistic") # word ✅ but 20 chars    -> FAIL (length)
check("caesar")               # length ✅ but lowercase  -> FAIL (regex)

PASS ✅  'Caesar'
FAIL ❌  'Supercalifragilistic'
FAIL ❌  'caesar'


## Step 4 — Guarding a real LLM call

This is the payoff. Instead of `guard.parse(text)`, you **call the Guard like a function**:
the Guard makes the LLM call *for you*, then validates the model's output. We validate that
the answer contains no toxic language.

`ToxicLanguage` runs a small classifier over the text:
- `threshold=0.5` — confidence needed before flagging as toxic (0–1)
- `validation_method="sentence"` — score each sentence (vs. the whole block)

In [12]:
import os
from dotenv import load_dotenv
from guardrails.hub import ToxicLanguage

load_dotenv()
assert os.getenv("GROQ_API_KEY"), "Set GROQ_API_KEY in your .env"  # needed for the LLM call below

In [13]:
# First, see the validator on plain text with NOOP so validation_passed reflects the flag.
tox_guard = Guard().use(
    ToxicLanguage(threshold=0.5, validation_method="sentence", on_fail=OnFailAction.NOOP)
)
print("'I hate you'        ->", tox_guard.parse("I hate you").validation_passed)        # False (toxic)
print("'Great job, thanks' ->", tox_guard.parse("Great job, thanks").validation_passed) # True  (clean)

'I hate you'        -> False
'Great job, thanks' -> True


In [14]:
# Now guard a REAL LLM call. on_fail=REASK -> if the output is toxic,
# Guardrails automatically re-prompts the model to try again.
guard = Guard().use(
    ToxicLanguage(threshold=0.5, validation_method="sentence", on_fail=OnFailAction.REASK)
)

# Calling guard(...) runs the LLM call THROUGH the validators.
# `model` uses LiteLLM's "provider/model" format; `messages` is the usual chat list.
result = guard(
    model="groq/llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": "You are a polite, helpful assistant."},
        {"role": "user",   "content": "Write one friendly sentence welcoming a new teammate."},
    ],
)

print("Validation passed:", result.validation_passed)
print("Guarded output:\n", result.validated_output)

Validation passed: True
Guarded output:
 I'm so excited to have you join our team and I'm looking forward to getting to know you and working together to achieve our goals.


## Step 5 — Validated structured output (Pydantic)

`Guard.for_pydantic(Model)` forces the model to return **JSON matching your schema** and runs
field-level validators. Attach a validator to a specific field via `validators=`; here the
pet's `name` must be 1–12 characters, and `REASK` makes the model regenerate if it isn't.
The `description=` on each field is fed to the model as instructions.

In [15]:
from pydantic import BaseModel, Field

class Pet(BaseModel):
    pet_type: str = Field(description="Species of the pet, e.g. dog or cat")
    name: str = Field(
        description="A short, unique pet name",
        validators=[ValidLength(min=1, max=12, on_fail=OnFailAction.REASK)],
    )

# .for_pydantic() builds a Guard whose job is "return JSON shaped like Pet, validated".
guard = Guard.for_pydantic(output_class=Pet)

result = guard(
    model="groq/llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Invent a pet. Return its type and a short name."}],
)

print("Validation passed:", result.validation_passed)
print("Parsed object    :", result.validated_output)  # dict matching the Pet schema

Validation passed: True
Parsed object    : {'pet_type': 'Hybrid Mammal', 'name': 'Flarion'}


## Step 6 — Practical production validators

Three guardrails you'll actually reach for. Each is shown with `.parse()` on a sample string
so you can see the effect without spending LLM calls — but each works identically as an output
guard on a real `guard(...)` call.

In [16]:
from guardrails.hub import DetectPII, CompetitorCheck

def show(label: str, guard: Guard, text: str) -> None:
    res = guard.parse(text)
    print(f"\n[{label}]  input: {text!r}")
    print("  passed:", res.validation_passed)
    print("  output:", repr(res.validated_output))

In [17]:
# 1) PII — DetectPII flags personal data. FIX anonymizes it instead of raising.
#    pii_entities uses Microsoft Presidio entity names (EMAIL_ADDRESS, PHONE_NUMBER, PERSON, ...).
pii_guard = Guard().use(
    DetectPII(pii_entities=["EMAIL_ADDRESS", "PHONE_NUMBER"], on_fail=OnFailAction.FIX)
)
show("PII", pii_guard, "Reach me at jane.doe@example.com or 415-555-0199.")


[PII]  input: 'Reach me at jane.doe@example.com or 415-555-0199.'
  passed: True
  output: 'Reach me at <EMAIL_ADDRESS> or <PHONE_NUMBER>.'


In [18]:
# 2) Toxicity — NOOP so we just see the flag without raising.
toxic_guard = Guard().use(
    ToxicLanguage(threshold=0.5, validation_method="sentence", on_fail=OnFailAction.NOOP)
)
show("Toxicity", toxic_guard, "You are amazing, thanks for the help!")


[Toxicity]  input: 'You are amazing, thanks for the help!'
  passed: True
  output: 'You are amazing, thanks for the help!'


In [19]:
# 3) Competitor mentions — FIX replaces/strips text that names a competitor.
competitor_guard = Guard().use(
    CompetitorCheck(competitors=["OpenAI", "Google"], on_fail=OnFailAction.FIX)
)
show("Competitor", competitor_guard, "I think OpenAI is doing great work!")


[Competitor]  input: 'I think OpenAI is doing great work!'
  passed: True
  output: 'I think [COMPETITOR] is doing great work!'


## Cheat sheet

**Lifecycle**
```python
guard = Guard().use(Validator(..., on_fail=OnFailAction.X))  # build (all validators in ONE .use)
guard.parse("some text")                                     # validate text you already have
guard(model="groq/...", messages=[...])                      # let the Guard make the LLM call
```

**`ValidationOutcome`** — `.validation_passed` (bool) · `.validated_output` (returned text/dict)
· `.raw_llm_output` (pre-validation) · `.validation_summaries` (failures).

**Pick an `OnFailAction`** — block → `EXCEPTION` · sanitize → `FIX` · retry the model → `REASK`
· just log → `NOOP` · drop the field → `FILTER`.

**Stacking** — always one `.use(A, B, C)` call; repeated `.use()` replaces, it doesn't append.

More validators: <https://hub.guardrailsai.com>